# SatDetection — Land Cover Segmentation
**CSCI 4366/6366 | The George Washington University**

Trains three models on LandCover.ai:
- Model A: SMP U-Net + ResNet-50, no pretraining
- Model B: SMP U-Net + ResNet-50, ImageNet pretrained
- Model C: Custom U-Net + ResNet-50, no pretraining

## 1. Setup

In [ ]:
# Speed settings for A100
import torch
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
if DEVICE == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
!pip install -q segmentation-models-pytorch torchgeo torchmetrics albumentations rasterio

In [ ]:
!git clone https://github.com/yrufai/SatDetection.git
%cd SatDetection

## 2. Data

In [ ]:
import img_helpers
from SegDataset import SegDataset
from TiledDataset import TiledDataset
from torchgeo.datasets import LandCoverAI
import os

# Download and tile the dataset (only needed once per session)
if not os.path.exists('data/tiles/images') or len(os.listdir('data/tiles/images')) == 0:
    print('Tiling dataset...')
    img_helpers.save_tiling()
else:
    print(f'Tiles already exist: {len(os.listdir("data/tiles/images"))} images')

In [ ]:
dataset = SegDataset(
    img_dir='data/tiles/images',
    mask_dir='data/tiles/masks',
)
print(f'Dataset size: {len(dataset)} tiles')

# Preview a few samples
for i in [0, 5, 10]:
    img_helpers.preview_img(dataset, i)

## 3. Training

In [ ]:
from SegModel import SegModel
import segmentation_models_pytorch as smp
from UNetResNet50 import UNetResNet50
import copy

EPOCHS     = 30
BATCH_SIZE = 128  # reduce to 64 or 32 if OOM
NUM_CLASSES = 5
TRAIN_SPLIT = 0.70
VAL_SPLIT   = 0.15

### Model A — SMP U-Net, no pretraining

In [ ]:
model_a = smp.Unet(
    encoder_name='resnet50',
    encoder_weights=None,
    in_channels=3,
    classes=NUM_CLASSES,
)

seg_a = SegModel(
    EPOCHS=EPOCHS,
    BATCH_SIZE=BATCH_SIZE,
    NUM_CLASSES=NUM_CLASSES,
    model=model_a,
    dataset=dataset,
    DEVICE=DEVICE,
)

print('=== Model A: SMP U-Net (no pretraining) ===')
seg_a.train(
    train_split=TRAIN_SPLIT,
    val_split=VAL_SPLIT,
    saved_model_name='best_model_smp_untrained',
)
print('\n--- Test ---')
seg_a.test()

### Model B — SMP U-Net, ImageNet pretrained

In [ ]:
model_b = smp.Unet(
    encoder_name='resnet50',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
)

seg_b = SegModel(
    EPOCHS=EPOCHS,
    BATCH_SIZE=BATCH_SIZE,
    NUM_CLASSES=NUM_CLASSES,
    model=model_b,
    dataset=dataset,
    DEVICE=DEVICE,
)

print('=== Model B: SMP U-Net (ImageNet pretrained) ===')
seg_b.train(
    train_split=TRAIN_SPLIT,
    val_split=VAL_SPLIT,
    saved_model_name='best_model_smp_pretrained',
)
print('\n--- Test ---')
seg_b.test()

### Model C — Custom U-Net, no pretraining

In [ ]:
model_c = UNetResNet50(num_classes=NUM_CLASSES, pretrained=False)

seg_c = SegModel(
    EPOCHS=EPOCHS,
    BATCH_SIZE=BATCH_SIZE,
    NUM_CLASSES=NUM_CLASSES,
    model=model_c,
    dataset=dataset,
    DEVICE=DEVICE,
)

print('=== Model C: Custom U-Net (no pretraining) ===')
seg_c.train(
    train_split=TRAIN_SPLIT,
    val_split=VAL_SPLIT,
    saved_model_name='best_model_custom_untrained',
)
print('\n--- Test ---')
seg_c.test()

## 4. Results & Visualization

In [ ]:
print('Loading example predictions...')
for name, seg in [('SMP Untrained', seg_a), ('SMP Pretrained', seg_b), ('Custom Untrained', seg_c)]:
    print(f'\n--- {name} ---')
    seg.load_example(5)

## 5. Save to Drive (optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os

DRIVE_DIR = '/content/drive/MyDrive/SatDetection'
os.makedirs(DRIVE_DIR, exist_ok=True)

for fname in os.listdir('models/'):
    shutil.copy(f'models/{fname}', f'{DRIVE_DIR}/{fname}')
    print(f'Saved {fname}')